# 02 - Dataset Binario, Balanceamento e Splits por Paciente

Este notebook implementa as tarefas 6, 7 e 8 da ordem recomendada:

1. Criar dataset binario limpo e realista.
2. Implementar balanceamento.
3. Criar splits por paciente.

O objetivo e transformar o NIH Chest X-rays em datasets controlados para o problema `No Finding` vs `Cardiomegaly`, evitando vazamento de paciente entre treino, validacao e teste.

## Pre-requisitos

Execute antes:

1. `notebooks/00_download_dataset_kaggle.ipynb`
2. `notebooks/01_eda_dataset.ipynb`

Este notebook espera encontrar `data/raw/Data_entry_2017.csv`. Se `data/raw/image_paths.csv` existir, os caminhos das imagens serao anexados aos CSVs gerados.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src import split_utils

config.ensure_project_directories()
config.seed_everything()

FIGURES_DIR = config.FIGURES_DIR
TABLES_DIR = config.TABLES_DIR
SPLITS_DIR = config.SPLITS_DIR

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.axisbelow"] = True

if HAS_SEABORN:
    sns.set_theme(style="whitegrid")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLITS_DIR:", SPLITS_DIR)

## Carregamento dos Metadados

Carregamos o CSV original do NIH e anexamos os caminhos locais das imagens quando `image_paths.csv` estiver disponivel.

In [ ]:
df_raw = split_utils.load_metadata()
df_raw = split_utils.attach_image_paths(df_raw)

print("Linhas originais:", len(df_raw))
print("Pacientes unicos:", df_raw["Patient ID"].nunique())
df_raw.head()

## Criacao dos Datasets Binarios

Criamos duas versoes:

### Versao limpa

- Classe 0: `No Finding` exatamente.
- Classe 1: `Cardiomegaly` exatamente.
- Exclui cardiomegalia acompanhada de outras patologias.

### Versao realista

- Classe 0: `No Finding` exatamente.
- Classe 1: qualquer imagem que contenha `Cardiomegaly`, mesmo com outras labels.

A versao limpa e melhor para interpretabilidade inicial. A versao realista representa melhor a natureza multi-label do dataset.

In [ ]:
dataset_clean, dataset_realistic = split_utils.create_binary_datasets(df_raw)

clean_path = SPLITS_DIR / "dataset_binary_clean.csv"
realistic_path = SPLITS_DIR / "dataset_binary_realistic.csv"

dataset_clean.to_csv(clean_path, index=False)
dataset_realistic.to_csv(realistic_path, index=False)

print("Dataset limpo salvo em:", clean_path)
print("Dataset realista salvo em:", realistic_path)
print("Linhas clean:", len(dataset_clean))
print("Linhas realistic:", len(dataset_realistic))

In [ ]:
dist_clean = split_utils.class_distribution(dataset_clean)
dist_realistic = split_utils.class_distribution(dataset_realistic)

distribution_summary = split_utils.write_summary_table(
    [("clean", dist_clean), ("realistic", dist_realistic)],
    TABLES_DIR / "splits_distribuicao_datasets_binarios.csv",
)

distribution_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_data = distribution_summary.copy()
plot_data["class"] = plot_data["binary_label_name"] + " (" + plot_data["source"] + ")"

if HAS_SEABORN:
    sns.barplot(data=plot_data, x="class", y="image_count", hue="source", dodge=False, ax=ax)
else:
    ax.bar(plot_data["class"], plot_data["image_count"])

ax.set_title("Distribuicao dos datasets binarios")
ax.set_xlabel("Classe e versao")
ax.set_ylabel("Quantidade de imagens")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "splits_distribuicao_datasets_binarios.png", dpi=160, bbox_inches="tight")
plt.show()

## Balanceamento

Implementamos duas informacoes para comparar estrategias:

1. CSV balanceado por undersampling.
2. Pesos de classe para treinar mantendo o dataset original.

O undersampling facilita uma primeira comparacao entre modelos. Os pesos de classe preservam mais dados e poderao ser usados no pipeline de treino.

In [ ]:
dataset_clean_balanced = split_utils.undersample_balance(dataset_clean)
dataset_realistic_balanced = split_utils.undersample_balance(dataset_realistic)

clean_balanced_path = SPLITS_DIR / "dataset_binary_clean_balanced_undersampled.csv"
realistic_balanced_path = SPLITS_DIR / "dataset_binary_realistic_balanced_undersampled.csv"

dataset_clean_balanced.to_csv(clean_balanced_path, index=False)
dataset_realistic_balanced.to_csv(realistic_balanced_path, index=False)

weights_clean = split_utils.compute_class_weights(dataset_clean)
weights_realistic = split_utils.compute_class_weights(dataset_realistic)

split_utils.save_class_weights(weights_clean, SPLITS_DIR / "class_weights_clean.json")
split_utils.save_class_weights(weights_realistic, SPLITS_DIR / "class_weights_realistic.json")

balanced_distribution_summary = split_utils.write_summary_table(
    [
        ("clean_original", split_utils.class_distribution(dataset_clean)),
        ("clean_balanced_undersampled", split_utils.class_distribution(dataset_clean_balanced)),
        ("realistic_original", split_utils.class_distribution(dataset_realistic)),
        ("realistic_balanced_undersampled", split_utils.class_distribution(dataset_realistic_balanced)),
    ],
    TABLES_DIR / "splits_distribuicao_balanceamento.csv",
)

print("Pesos clean:", weights_clean)
print("Pesos realistic:", weights_realistic)
balanced_distribution_summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_data = balanced_distribution_summary.copy()
plot_data["class"] = plot_data["binary_label_name"]

if HAS_SEABORN:
    sns.barplot(data=plot_data, x="source", y="image_count", hue="class", ax=ax)
else:
    for class_name, group in plot_data.groupby("class"):
        ax.bar(group["source"], group["image_count"], label=class_name, alpha=0.75)
    ax.legend()

ax.set_title("Distribuicao antes e depois do balanceamento")
ax.set_xlabel("Dataset")
ax.set_ylabel("Quantidade de imagens")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "splits_distribuicao_balanceamento.png", dpi=160, bbox_inches="tight")
plt.show()

## Splits por Paciente

O mesmo paciente pode ter varias imagens. Por isso, fazemos split por `Patient ID`, nao por linha individual. Isso reduz vazamento de dados entre treino, validacao e teste.

Proporcao alvo:

- 70% treino
- 15% validacao
- 15% teste

Geramos splits para as versoes clean e realistic balanceadas. Para os arquivos canonicos `train.csv`, `val.csv` e `test.csv`, usamos a versao realista balanceada por ser a mais proxima do carater multi-label do NIH.

In [ ]:
clean_patient_splits = split_utils.split_patient_ids(dataset_clean_balanced)
realistic_patient_splits = split_utils.split_patient_ids(dataset_realistic_balanced)

clean_split_frames = split_utils.apply_patient_split(dataset_clean_balanced, clean_patient_splits)
realistic_split_frames = split_utils.apply_patient_split(dataset_realistic_balanced, realistic_patient_splits)

split_utils.save_split_frames(clean_split_frames, SPLITS_DIR, prefix="clean")
split_utils.save_split_frames(realistic_split_frames, SPLITS_DIR, prefix="realistic")

# Arquivos canonicos usados pelas proximas etapas de treino.
canonical_paths = split_utils.save_split_frames(realistic_split_frames, SPLITS_DIR, prefix=None)

print("Arquivos canonicos:")
for split_name, path in canonical_paths.items():
    print(split_name, "->", path)

In [ ]:
clean_split_summary = split_utils.validate_patient_split(clean_split_frames)
realistic_split_summary = split_utils.validate_patient_split(realistic_split_frames)

split_summary = split_utils.write_summary_table(
    [("clean_balanced", clean_split_summary), ("realistic_balanced", realistic_split_summary)],
    TABLES_DIR / "splits_resumo_por_paciente.csv",
)

split_summary

In [ ]:
def patient_overlap_count(left: pd.DataFrame, right: pd.DataFrame) -> int:
    return len(set(left["Patient ID"].unique()) & set(right["Patient ID"].unique()))


overlap_checks = []
for dataset_name, frames in [("clean", clean_split_frames), ("realistic", realistic_split_frames)]:
    overlap_checks.extend(
        [
            {
                "dataset": dataset_name,
                "pair": "train_val",
                "overlapping_patients": patient_overlap_count(frames["train"], frames["val"]),
            },
            {
                "dataset": dataset_name,
                "pair": "train_test",
                "overlapping_patients": patient_overlap_count(frames["train"], frames["test"]),
            },
            {
                "dataset": dataset_name,
                "pair": "val_test",
                "overlapping_patients": patient_overlap_count(frames["val"], frames["test"]),
            },
        ]
    )

overlap_checks_df = pd.DataFrame(overlap_checks)
overlap_checks_df.to_csv(TABLES_DIR / "splits_validacao_vazamento_paciente.csv", index=False)
overlap_checks_df

## Arquivos Gerados

Datasets:

- `data/splits/dataset_binary_clean.csv`
- `data/splits/dataset_binary_realistic.csv`
- `data/splits/dataset_binary_clean_balanced_undersampled.csv`
- `data/splits/dataset_binary_realistic_balanced_undersampled.csv`

Pesos de classe:

- `data/splits/class_weights_clean.json`
- `data/splits/class_weights_realistic.json`

Splits especificos:

- `data/splits/train_clean.csv`, `val_clean.csv`, `test_clean.csv`
- `data/splits/train_realistic.csv`, `val_realistic.csv`, `test_realistic.csv`

Splits canonicos para as proximas etapas:

- `data/splits/train.csv`
- `data/splits/val.csv`
- `data/splits/test.csv`

Tabelas e figuras de validacao foram salvas em `reports/`.

## Proxima Etapa

Depois de validar os splits, seguir para as tarefas 9 e 10:

1. Implementar pre-processamento das imagens.
2. Criar Dataset e DataLoader em PyTorch.